In [10]:
import pandas as pd


In [14]:
df = pd.read_csv('Cleaned_Dataset.csv')
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8168 entries, 0 to 8167
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             8168 non-null   int64  
 1   id                     8168 non-null   str    
 2   event_type             8168 non-null   str    
 3   latitude               8168 non-null   float64
 4   longitude              8168 non-null   float64
 5   endlatitude            8168 non-null   float64
 6   endlongitude           8168 non-null   float64
 7   address                8168 non-null   str    
 8   end_address            8168 non-null   str    
 9   event_cause            8168 non-null   str    
 10  requires_road_closure  8168 non-null   bool   
 11  start_datetime         8168 non-null   str    
 12  end_datetime           8168 non-null   str    
 13  status                 8168 non-null   str    
 14  authenticated          8168 non-null   str    
 15  modified_dateti

## Remove unnecessary columns 

In [40]:
drop_cols = [
    'id',
    'client_id',
    'created_by_id',
    'last_modified_by_id',
    'authenticated',
    'description',
]
df.drop(columns=drop_cols, inplace=True)


Changing the data column into proper format for better analysis

In [16]:
datetime_cols = [
    'start_datetime',
    'end_datetime',
    'closed_datetime',
    'modified_datetime',
    'created_date'
]

for col in datetime_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors='coerce',
        format='mixed'
    )

How long did the disruption last?

In [17]:
df['event_duration_min'] = (
    df['end_datetime'] -
    df['start_datetime']
).dt.total_seconds()/60


Closure Delay

In [19]:
df['closure_delay_min'] = (
    df['closed_datetime'] -
    df['end_datetime']
).dt.total_seconds()/60

Hour

In [20]:
df['hour'] = df['start_datetime'].dt.hour

Day of Week

In [21]:
df['day_of_week'] = df['start_datetime'].dt.dayofweek

Weekend Flag

In [22]:
df['is_weekend'] = (
    df['day_of_week'] >= 5
).astype(int)


Month

In [25]:
df['month'] = df['start_datetime'].dt.month


Peak Hour

In [26]:
df['peak_hour'] = (
    df['hour'].between(7,10) |
    df['hour'].between(16,20)
).astype(int)

here we are seeing the area affected in the event

In [28]:
from geopy.distance import geodesic

def distance_km(row):
    return geodesic(
        (row['latitude'], row['longitude']),
        (row['endlatitude'], row['endlongitude'])
    ).km

df['stretch_length_km'] = df.apply(
    distance_km,
    axis=1
)

here we measure wheter the event is moving or not 

In [29]:
def resolved_distance(row):
    return geodesic(
        (row['latitude'], row['longitude']),
        (
            row['resolved_at_latitude'],
            row['resolved_at_longitude']
        )
    ).km

df['resolution_shift_km'] = df.apply(
    resolved_distance,
    axis=1
)

How often does this corridor experience events?

In [30]:
corridor_freq = (
    df['corridor']
    .value_counts()
)
df['corridor_event_count'] = (
    df['corridor']
    .map(corridor_freq)
)

Corridor Risk Score

In [31]:
closure_rate = (
    df.groupby('corridor')
      ['requires_road_closure']
      .mean()
)

Junction Event Count

In [36]:
junction_freq = (
    df['junction']
      .value_counts()
)
df['junction_event_count'] = (
    df['junction']
      .map(junction_freq)
)

Junction Priority Rate

In [37]:
priority_rate = (
    df.groupby('junction')
      ['priority']
      .apply(
          lambda x:(x=='High').mean()
      )
)

Vehicle Severity Score

In [38]:
severity_map = {
    'two_wheeler':1,
    'auto':2,
    'private_car':2,
    'lcv':3,
    'bmtc_bus':4,
    'heavy_vehicle':5,
    'others':1
}

df['vehicle_severity'] = (
    df['veh_type']
      .map(severity_map)
)

Cargo Presence

In [39]:
df['has_cargo'] = (
    df['cargo_material']
    != 'no goods'
).astype(int)

Create Cause Severity Score

In [41]:
def cause_group(cause):

    cause = str(cause).lower()

    if 'accident' in cause:
        return 'Accident'

    elif 'breakdown' in cause:
        return 'Breakdown'

    elif 'water' in cause or 'flood' in cause:
        return 'Flooding'

    elif 'signal' in cause:
        return 'Signal Failure'

    elif 'pothole' in cause or 'road' in cause:
        return 'Road Damage'

    else:
        return 'Other'

df['cause_group'] = df['event_cause'].apply(cause_group)
cause_score = {
    'Road Damage':1,
    'Signal Failure':2,
    'Breakdown':3,
    'Accident':4,
    'Flooding':5,
    'Other':2
}

df['cause_severity'] = (
    df['cause_group']
      .map(cause_score)
)

In [44]:
df['diversion_required'] = (
    (df['requires_road_closure'] == True) |
    (df['priority'] == 'High')
).astype(int)

In [45]:
print(df['priority'].value_counts())

priority
High    4754
Low     2948
Name: count, dtype: int64


In [50]:
def manpower_category(row):

    score = 0

    # Priority
    if row['priority'] == 'High':
        score += 3
    else:
        score += 1

    # Road Closure
    if row['requires_road_closure']:
        score += 2

    # Vehicle Severity
    if row['vehicle_severity'] >= 4:
        score += 2

    # Peak Hour
    if row['peak_hour'] == 1:
        score += 1

    # Long Duration
    if row['event_duration_min'] > 60:
        score += 1

    return score

df['manpower_score'] = df.apply(
    manpower_category,
    axis=1
)

In [51]:
df['manpower_score'].value_counts().sort_index()

manpower_score
1     952
2     661
3    2347
4    1672
5    1240
6     680
7     121
8      25
9       4
Name: count, dtype: int64

In [53]:
def manpower_label(score):

    if score <= 3:
        return 'Low'

    elif score <= 5:
        return 'Medium'

    else:
        return 'High'

df['manpower_required'] = (
    df['manpower_score']
      .apply(manpower_label)
)
df['manpower_required'].value_counts()

manpower_required
Low       3960
Medium    2912
High       830
Name: count, dtype: int64

In [54]:
df['requires_road_closure'].value_counts()

requires_road_closure
False    7193
True      509
Name: count, dtype: int64

In [55]:
df['manpower_required'].value_counts()

manpower_required
Low       3960
Medium    2912
High       830
Name: count, dtype: int64

In [49]:
df['priority'].value_counts()

priority
High    4754
Low     2948
Name: count, dtype: int64

## Encode Categorical Variables

In [57]:
from sklearn.preprocessing import LabelEncoder

# Targets
target_cols = [
    'priority',
    'requires_road_closure',
    'diversion_required',
    'manpower_required'
]

# Features
X = df.drop(columns=target_cols)

# Encode categorical columns once
encoders = {}

for col in X.select_dtypes(include='object').columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = le

print("Feature matrix shape:", X.shape)

Feature matrix shape: (7702, 45)


C:\Users\91947\AppData\Local\Temp\ipykernel_26584\787283439.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include='object').columns:


Removed duplicate values 

In [75]:
df = df.drop_duplicates(
    subset=[
        'event_type',
        'event_cause',
        'address',
        'start_datetime'
    ]
)

In [88]:
print(X.select_dtypes(include='object').columns.tolist())

['event_type', 'address', 'end_address', 'event_cause', 'status', 'veh_type', 'corridor', 'cargo_material', 'reason_breakdown', 'police_station', 'gba_identifier', 'zone', 'junction', 'description_clean', 'cause_group']


C:\Users\91947\AppData\Local\Temp\ipykernel_26584\1870110945.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(X.select_dtypes(include='object').columns.tolist())


In [90]:
print(
    X.select_dtypes(
        include=['object', 'string']
    ).columns.tolist()
)

[]


verify feature importance

In [72]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': priority_model.feature_importances_
})

importance.sort_values(
    by='Importance',
    ascending=False
).head(20)

,Feature,Importance
33,corridor_event_count,0.523140
11,corridor,0.186668
39,manpower_score,0.132212
31,stretch_length_km,0.021882
17,resolved_at_longitude,0.017624
3,longitude,0.014952
35,vehicle_severity,0.011978
14,police_station,0.010866
2,latitude,0.009382
10,veh_type,0.009363


In [73]:
df.duplicated().sum()

np.int64(0)

In [74]:
df.duplicated(subset=[
    'event_type',
    'event_cause',
    'address',
    'start_datetime'
]).sum()

np.int64(76)

## Create early warning features

In [ ]:
early_features = [

    # Incident Information
    'event_type',
    'event_cause',
    'veh_type',

    # Location
    'latitude',
    'longitude',

    # Time Features
    'hour',
    'peak_hour',
    'month',
    'is_weekend',

    # Engineered Severity Features
    'vehicle_severity',
    'cause_severity'
]

In [111]:
X_early = df[early_features].copy()

## Label encoding 

In [112]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in X_early.select_dtypes(include=['object', 'string']).columns:

    le = LabelEncoder()

    X_early[col] = le.fit_transform(
        X_early[col].astype(str)
    )

    label_encoders[col] = le

print("Encoding Complete")

Encoding Complete


In [113]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

def train_model(X, y, model_name):

    print("\n" + "="*60)
    print(f"MODEL: {model_name}")
    print("="*60)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced'
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("\nAccuracy:")
    print(accuracy_score(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model

In [114]:
y_priority = df['priority']

In [115]:
priority_model = train_model(
    X_early,
    y_priority,
    "Priority Early Warning"
)


MODEL: Priority Early Warning

Accuracy:
0.8905635648754915

Classification Report:
              precision    recall  f1-score   support

        High       0.88      0.95      0.92       947
         Low       0.91      0.79      0.85       579

    accuracy                           0.89      1526
   macro avg       0.90      0.87      0.88      1526
weighted avg       0.89      0.89      0.89      1526


Confusion Matrix:
[[901  46]
 [121 458]]


In [116]:
y_barricade = (
    df['requires_road_closure']
    .astype(int)
)

In [117]:
barricade_model = train_model(
    X_early,
    y_barricade,
    "Barricade Early Warning"
)


MODEL: Barricade Early Warning

Accuracy:
0.936435124508519

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.97      1424
           1       0.59      0.17      0.26       102

    accuracy                           0.94      1526
   macro avg       0.76      0.58      0.61      1526
weighted avg       0.92      0.94      0.92      1526


Confusion Matrix:
[[1412   12]
 [  85   17]]


In [118]:
y_diversion = df['diversion_required']

In [119]:
diversion_model = train_model(
    X_early,
    y_diversion,
    "Diversion Early Warning"
)


MODEL: Diversion Early Warning

Accuracy:
0.8702490170380078

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.71      0.79       523
           1       0.86      0.95      0.91      1003

    accuracy                           0.87      1526
   macro avg       0.88      0.83      0.85      1526
weighted avg       0.87      0.87      0.87      1526


Confusion Matrix:
[[372 151]
 [ 47 956]]


In [120]:
y_manpower = df['manpower_required']

In [121]:
manpower_model = train_model(
    X_early,
    y_manpower,
    "Manpower Early Warning"
)


MODEL: Manpower Early Warning

Accuracy:
0.7896461336828309

Classification Report:
              precision    recall  f1-score   support

        High       0.76      0.66      0.71       166
         Low       0.85      0.85      0.85       781
      Medium       0.72      0.74      0.73       579

    accuracy                           0.79      1526
   macro avg       0.78      0.75      0.76      1526
weighted avg       0.79      0.79      0.79      1526


Confusion Matrix:
[[110   6  50]
 [  2 664 115]
 [ 33 115 431]]


Feature Importance Function

In [122]:
import pandas as pd

def show_feature_importance(model, X):

    importance = pd.DataFrame({

        'Feature': X.columns,

        'Importance':
        model.feature_importances_

    })

    importance = importance.sort_values(
        by='Importance',
        ascending=False
    )

    return importance

Priority importance

In [123]:
show_feature_importance(
    priority_model,
    X_early
).head(10)

,Feature,Importance
4,longitude,0.450069
3,latitude,0.362770
6,month,0.056053
1,event_cause,0.034592
2,veh_type,0.028457
8,vehicle_severity,0.018471
7,is_weekend,0.017362
5,peak_hour,0.016711
9,cause_severity,0.014504
0,event_type,0.001011


Barricade importance 

In [124]:
show_feature_importance(
    barricade_model,
    X_early
).head(10)

,Feature,Importance
3,latitude,0.315389
4,longitude,0.310866
6,month,0.091968
1,event_cause,0.087166
9,cause_severity,0.049497
8,vehicle_severity,0.042456
2,veh_type,0.042160
5,peak_hour,0.028518
7,is_weekend,0.026878
0,event_type,0.005104


Diversion Importance 

In [125]:
show_feature_importance(
    diversion_model,
    X_early
).head(10)

,Feature,Importance
4,longitude,0.439168
3,latitude,0.364641
6,month,0.060692
1,event_cause,0.033262
2,veh_type,0.028063
7,is_weekend,0.019213
5,peak_hour,0.018837
8,vehicle_severity,0.018655
9,cause_severity,0.015573
0,event_type,0.001897


ManPower Importance 

In [126]:
show_feature_importance(
    manpower_model,
    X_early
).head(10)

,Feature,Importance
4,longitude,0.259691
3,latitude,0.241496
5,peak_hour,0.159011
8,vehicle_severity,0.126149
2,veh_type,0.088016
6,month,0.057117
1,event_cause,0.034549
7,is_weekend,0.016254
9,cause_severity,0.015509
0,event_type,0.002207


## Operational Models 
basically in this model we get more info about the event to take more precise decision

In [127]:
operational_features = [

    'event_type',
    'event_cause',
    'veh_type',

    'corridor',
    'police_station',
    'zone',
    'junction',

    'latitude',
    'longitude',

    'hour',
    'peak_hour',
    'month',
    'is_weekend',

    'vehicle_severity',
    'cause_severity',

    'stretch_length_km',

    'corridor_event_count',
    'junction_event_count'
]

In [128]:
X_operational = df[operational_features].copy()

In [129]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in X_operational.select_dtypes(
    include=['object','string']
).columns:

    le = LabelEncoder()

    X_operational[col] = le.fit_transform(
        X_operational[col].astype(str)
    )

    label_encoders[col] = le

In [130]:
y_priority = df['priority']

priority_operational_model = train_model(
    X_operational,
    y_priority,
    "Operational Priority Model"
)


MODEL: Operational Priority Model

Accuracy:
0.9980340760157274

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00       947
         Low       0.99      1.00      1.00       579

    accuracy                           1.00      1526
   macro avg       1.00      1.00      1.00      1526
weighted avg       1.00      1.00      1.00      1526


Confusion Matrix:
[[944   3]
 [  0 579]]


In [131]:
y_barricade = (
    df['requires_road_closure']
    .astype(int)
)

barricade_operational_model = train_model(
    X_operational,
    y_barricade,
    "Operational Barricade Model"
)


MODEL: Operational Barricade Model

Accuracy:
0.9967234600262124

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1424
           1       0.96      0.99      0.98       102

    accuracy                           1.00      1526
   macro avg       0.98      0.99      0.99      1526
weighted avg       1.00      1.00      1.00      1526


Confusion Matrix:
[[1420    4]
 [   1  101]]


In [132]:
y_diversion = (
    df['diversion_required']
)

diversion_operational_model = train_model(
    X_operational,
    y_diversion,
    "Operational Diversion Model"
)


MODEL: Operational Diversion Model

Accuracy:
0.9980340760157274

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       523
           1       1.00      1.00      1.00      1003

    accuracy                           1.00      1526
   macro avg       1.00      1.00      1.00      1526
weighted avg       1.00      1.00      1.00      1526


Confusion Matrix:
[[ 522    1]
 [   2 1001]]


In [133]:
y_manpower = (
    df['manpower_required']
)

manpower_operational_model = train_model(
    X_operational,
    y_manpower,
    "Operational Manpower Model"
)


MODEL: Operational Manpower Model

Accuracy:
0.8951507208387942

Classification Report:
              precision    recall  f1-score   support

        High       0.93      0.77      0.84       166
         Low       0.90      0.96      0.93       781
      Medium       0.87      0.84      0.86       579

    accuracy                           0.90      1526
   macro avg       0.90      0.86      0.88      1526
weighted avg       0.90      0.90      0.89      1526


Confusion Matrix:
[[127   0  39]
 [  0 750  31]
 [  9  81 489]]
